# 04 - Analyse SQL

Objectif : structurer les données dans SQLite et valider les KPI avec SQL. Le dashboard Power BI utilise directement `incident_clean.csv` et des mesures DAX dynamiques.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DATABASE_DIR = PROJECT_ROOT / 'data' / 'database'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
DATA_PATH = PROCESSED_DIR / 'incident_clean.csv'
DB_PATH = DATABASE_DIR / 'incident.db'

df = pd.read_csv(DATA_PATH, low_memory=False)
conn = sqlite3.connect(DB_PATH)
df.to_sql('incidents', conn, if_exists='replace', index=False)

print(f'Base créée : {DB_PATH}')

In [ ]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

# Les résultats SQL restent dans le notebook : aucun CSV de synthèse n'est requis.
# Power BI calcule les mêmes KPI depuis incident_clean.csv avec des mesures DAX.

## KPI globaux

In [ ]:
kpi_sql = run_sql('''
SELECT
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h,
    ROUND(AVG(closure_time_hours), 2) AS cloture_moyenne_h,
    ROUND(AVG(reassignment_count), 2) AS reassignations_moyennes,
    ROUND(AVG(reopen_count), 2) AS reouvertures_moyennes
FROM incidents;
''')

kpi_sql

## Performance par groupe support

In [ ]:
groupe_sql = run_sql('''
SELECT
    assignment_group,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h
FROM incidents
GROUP BY assignment_group
HAVING COUNT(*) >= 20
ORDER BY incidents DESC
LIMIT 10;
''')

groupe_sql

## SLA par priorité

In [ ]:
priorite_sql = run_sql('''
SELECT
    priority,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h
FROM incidents
GROUP BY priority
ORDER BY priority;
''')

priorite_sql

## Volume mensuel

In [ ]:
mois_sql = run_sql('''
SELECT
    opened_month,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct
FROM incidents
GROUP BY opened_month
ORDER BY opened_month;
''')

mois_sql

## Effet des réassignations

In [ ]:
reassignment_sql = run_sql('''
SELECT
    CASE WHEN has_reassignment = 1 THEN 'Avec réassignation' ELSE 'Sans réassignation' END AS reassignment_status,
    COUNT(*) AS incidents,
    ROUND(AVG(sla_compliant) * 100, 2) AS sla_pct,
    ROUND(AVG(resolution_time_hours), 2) AS resolution_moyenne_h
FROM incidents
GROUP BY has_reassignment
ORDER BY has_reassignment;
''')

reassignment_sql

## Incidents les plus longs

Cette requête permet d'identifier les incidents avec les temps de résolution les plus élevés.

In [ ]:
top_long_incidents_sql = run_sql('''
SELECT
    number,
    assignment_group,
    category,
    priority,
    resolution_time_hours
FROM incidents
WHERE resolution_time_hours IS NOT NULL
ORDER BY resolution_time_hours DESC
LIMIT 25;
''')

top_long_incidents_sql

In [ ]:
conn.close()

## Conclusion

Les requêtes SQL fournissent un contrôle structuré des indicateurs IT calculés dans Power BI.

Les résultats confirment trois axes de pilotage : surveiller le SLA global, isoler les priorités critiques et hautes dont la conformité SLA est très faible, et suivre les groupes support qui combinent volume élevé et délais de résolution importants. La tendance mensuelle doit être interprétée avec prudence, car la période observée est très déséquilibrée en volume.